# Experiment 02 — Deterministic Baseline (Linear Regression, LSTM & BiLSTM, Uni & Multi)

**Goal:** Compare **Linear Regression**, **LSTM** and **BiLSTM** using both **Univariate** and **Multivariate** features, saving detailed per-window metrics.

In [ ]:
# ── Cell 1: Environment Setup ────────────────────────────────────
from pathlib import Path
import subprocess, sys

PROJECT_ROOT = Path("/kaggle/working/stlf-entso-2026")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/AlvinHarist/stlf-entso-2026.git",
         str(PROJECT_ROOT)],
        check=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# ── Cell 2: Imports & Configuration ──────────────────────────────
import json
import yaml
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from src.utils.seed import set_seed
from src.data.load_data import load_dataset
from src.data.preprocessing import chronological_split, fit_preprocessor, transform_data, inverse_y
from src.data.windowing import create_train_windows, create_evaluation_windows
from src.models.lstm import build_lstm
from src.models.bilstm import build_bilstm
from src.training.trainer import train_model
from src.evaluation.point_metrics import compute_all_metrics, mae, rmse, mape, smape

CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

SEED       = config["seed"]
LOOKBACK   = config["windowing"]["lookback"]
HORIZON    = config["windowing"]["horizon"]
TARGET_COL = config["data"]["target_col"]
WEATHER    = config["data"]["weather_features"]
UNITS      = config["model"]["units"]
DROPOUT    = config["model"]["dropout"]
LR         = config["model"]["learning_rate"]
EPOCHS     = config["training"]["epochs"]
BATCH_SIZE = config["training"]["batch_size"]
PATIENCE   = config["training"]["patience"]

# Kaggle data path
KAGGLE_DATA_DIR = Path("/kaggle/input/stlf-entso-2026")
DATA_PATH = None
if KAGGLE_DATA_DIR.exists():
    for p in KAGGLE_DATA_DIR.rglob("*.csv"):
        if "combined_AT" in p.name:
            DATA_PATH = p
            break
if DATA_PATH is None:
    fallback = PROJECT_ROOT / config["data"]["path"]
    if fallback.exists():
        DATA_PATH = fallback
if DATA_PATH is None:
    fallback = PROJECT_ROOT / "df_combined_AT.csv"
    if fallback.exists():
        DATA_PATH = fallback
if DATA_PATH is None:
    raise FileNotFoundError("Cannot locate df_combined_AT.csv")

RESULTS_DIR = PROJECT_ROOT / "results" / "deterministic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Cell 3: Load Data ────────────────────────────────────────────
df = load_dataset(DATA_PATH, timestamp_col=config["data"]["timestamp_col"])
train_df, val_df, test_df = chronological_split(
    df, config["split"]["train_ratio"], config["split"]["val_ratio"]
)

In [ ]:
# ── Cell 4: Experiment Loop ──────────────────────────────────────
models_to_test = ["LinearRegression", "LSTM", "BiLSTM"]
features_to_test = ["univariate", "multivariate"]

all_results = {}

for model_name in models_to_test:
    for feat_type in features_to_test:
        print(f"\n{'='*60}")
        print(f" Running {model_name} with {feat_type.upper()} features")
        print(f"{'='*60}")
        
        # 1. Features Setup
        if feat_type == "univariate":
            feature_cols = [TARGET_COL]
            use_yj = False
        else:
            feature_cols = [TARGET_COL] + WEATHER
            use_yj = config["preprocessing"]["use_yeojohnson"]
            
        skewed = config["preprocessing"]["skewed_cols"] if use_yj else None
        
        set_seed(SEED)
        
        # 2. Preprocessing
        preprocessor = fit_preprocessor(
            train_df, TARGET_COL, feature_cols,
            skewed_cols=skewed, use_yeojohnson=use_yj
        )
        
        X_tr, y_tr = transform_data(train_df, preprocessor)
        X_va, y_va = transform_data(val_df, preprocessor)
        X_te, y_te = transform_data(test_df, preprocessor)
        
        Xw_tr, yw_tr = create_train_windows(X_tr, y_tr, LOOKBACK, HORIZON)
        Xw_va, yw_va = create_evaluation_windows(X_va, y_va, X_tr, y_tr, LOOKBACK, HORIZON)
        Xw_te, yw_te = create_evaluation_windows(X_te, y_te, X_va, y_va, LOOKBACK, HORIZON)
        
        # 3. Build & Train Model
        if model_name == "LinearRegression":
            # Flatten inputs for scikit-learn (N, lookback * features)
            X_tr_flat = Xw_tr.reshape(Xw_tr.shape[0], -1)
            X_te_flat = Xw_te.reshape(Xw_te.shape[0], -1)
            
            model = LinearRegression()
            model.fit(X_tr_flat, yw_tr)
            pred_scaled = model.predict(X_te_flat)
            best_epoch = "N/A"
        else:
            n_features = Xw_tr.shape[2]
            if model_name == "LSTM":
                model = build_lstm(LOOKBACK, n_features, HORIZON, UNITS, DROPOUT, LR)
            else:
                model = build_bilstm(LOOKBACK, n_features, HORIZON, UNITS, DROPOUT, LR)
                
            tr_res = train_model(model, Xw_tr, yw_tr, Xw_va, yw_va, EPOCHS, BATCH_SIZE, PATIENCE, verbose=1)
            pred_scaled = model.predict(Xw_te)
            best_epoch = tr_res["best_epoch"]
        
        # 4. Inverse Transform
        pred_mw = inverse_y(pred_scaled, preprocessor)
        actual_mw = inverse_y(yw_te, preprocessor)
        
        # 5. Aggregate Metrics
        mets = compute_all_metrics(actual_mw, pred_mw)
        all_results[f"{model_name}_{feat_type}"] = mets
        
        print(f"\nTest Metrics for {model_name} ({feat_type}):")
        for k, v in mets.items():
            print(f"  {k}: {v:.4f}")
            
        # 6. Per-Window Metrics & Saving
        N, H = pred_mw.shape
        per_window_results = []
        
        for i in range(N):
            y_true_i = actual_mw[i]
            y_pred_i = pred_mw[i]
            row_data = {
                "window_index": i,
                "mae": mae(y_true_i, y_pred_i),
                "rmse": rmse(y_true_i, y_pred_i),
                "mape": mape(y_true_i, y_pred_i),
                "smape": smape(y_true_i, y_pred_i)
            }
            for h in range(H):
                row_data[f"actual_h{h+1}"] = y_true_i[h]
                row_data[f"pred_h{h+1}"] = y_pred_i[h]
            per_window_results.append(row_data)
            
        df_per_window = pd.DataFrame(per_window_results)
        csv_path = RESULTS_DIR / f"{model_name}_{feat_type}_per_window.csv"
        df_per_window.to_csv(csv_path, index=False)
        print(f"Per-window metrics saved to {csv_path}")
        
        # 7. Save JSON record
        rec = {
            "experiment": "02_deterministic_baseline",
            "model": model_name,
            "features": feat_type,
            "lookback": LOOKBACK,
            "horizon": HORIZON,
            "test_metrics": mets,
            "best_epoch": best_epoch
        }
        json_path = RESULTS_DIR / f"{model_name}_{feat_type}_{HORIZON}h.json"
        with open(json_path, "w") as f:
            json.dump(rec, f, indent=2)


In [ ]:
# ── Cell 5: Summary Table ────────────────────────────────────────
print("\n" + "=" * 65)
print("EXPERIMENT RESULTS SUMMARY")
print("=" * 65)
print(f"{'Model & Features':<25s} {'MAE':>9s} {'RMSE':>9s} {'MAPE':>9s} {'sMAPE':>9s}")
print("-" * 65)
for name, m in all_results.items():
    print(f"{name:<25s} {m['MAE']:>9.2f} {m['RMSE']:>9.2f} {m['MAPE']:>9.2f} {m['sMAPE']:>9.2f}")
